In [2]:
# ────────────────────────────────────────────────────────────────
# Copy & paste everything from here ↓
# ────────────────────────────────────────────────────────────────

import sys
import os
from pathlib import Path

# 1) Locate this script (or notebook) directory
try:
    script_dir = Path(__file__).resolve().parent
except NameError:
    # __file__ doesn't exist in notebooks or REPLs
    script_dir = Path.cwd()

# 2) Assume project root is one level up from `python/`
project_root = script_dir.parent

# 3) Sanity check: ensure there's a `data/` folder at the root
if not (project_root / "data").is_dir():
    raise RuntimeError(f"Project root {project_root!r} has no data/ folder.")

# 4) Prepend to sys.path so you can `import` anywhere in Music_Project
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Now you can freely do:
#   import pandas as pd
#   df = pd.read_csv(project_root / "data" / "clean" / "covers_clean.csv")
# ────────────────────────────────────────────────────────────────
# Copy & paste everything above ↑
# ────────────────────────────────────────────────────────────────

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import wikipediaapi
from concurrent.futures import ProcessPoolExecutor, as_completed
from concurrent.futures import ThreadPoolExecutor
import threading
import time
import ast
from collections import Counter

# Load data
df_orig = pd.read_csv(project_root / "data" / "raw" / "originals.csv")
df_cov = pd.read_csv(project_root / "data" / "raw" / "covers.csv")
df_neo = pd.read_csv("https://raw.githubusercontent.com/freiraum-bq/Music_Project/main/data/raw/neo4j_artists.csv")

# The Wikapedia Part of the Graph
requires: neo4j data import

In [35]:
# Loading Wikapedia
wiki = wikipediaapi.Wikipedia(user_agent="MusicGraphExample (research@example.com)", language="en")

# Helpful Dataframes
df_neo_artists = df_neo[['artist_id', 'common_name', 'wiki_url']]
df_neo_artists = df_neo_artists[df_neo_artists['wiki_url'].notna() & (df_neo_artists['wiki_url'] != '')]

artist_urls = dict(zip(df_neo_artists['artist_id'], df_neo_artists['wiki_url']))
url_to_artist = {url: id for id, url in artist_urls.items()}
id_to_name = dict(zip(df_neo_artists['artist_id'], df_neo_artists['common_name']))

def get_page_links(url):
    tries = 3
    wiki = wikipediaapi.Wikipedia(user_agent="MusicGraphExample (research@example.com)", language="en")
    for attempt in range(tries):
        try:
            if '/wiki/' not in url:
                return url, set()
            page_title = url.split('/wiki/')[-1]
            page = wiki.page(page_title)
            if not page.exists():
                return url, set()
            links = page.links.keys()
            full_urls = {f"https://en.wikipedia.org/wiki/{link}" for link in links}
            return url, full_urls
        except Exception as e:
            if attempt == tries - 1:
                print(f"Exception in get_page_links for url {url}: {e}")
                return url, set()
            else:
                time.sleep(2 ** attempt)

def build_graph_threaded(artist_urls, url_to_artist, id_to_name, max_workers=10, batch_print=200):
    G = nx.DiGraph()
    artist_ids = list(artist_urls.keys())
    total = len(artist_ids)
    lock = threading.Lock()
    progress = {'count': 0}

    for artist in artist_ids:
        G.add_node(artist, name=id_to_name.get(artist, "Unknown"), genre = "sth")

    def worker(artist):
        url = artist_urls[artist]
        url, linked_urls = get_page_links(url)
        edges = []
        for linked_url in linked_urls:
            if linked_url in url_to_artist and linked_url != url:
                mentioned_artist = url_to_artist[linked_url]
                edges.append((artist, mentioned_artist))
        with lock:
            progress['count'] += 1
            if progress['count'] % batch_print == 0 or progress['count'] == total:
                print(f"Processed {progress['count']} / {total} artists...")
        return edges

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = executor.map(worker, artist_ids)

        for edges in results:
            for u, v in edges:
                G.add_edge(u, v, relation={'MENTIONS'})

    print(f"Graph built with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")
    return G

# Usage example:
G = build_graph_threaded(artist_urls, url_to_artist, id_to_name)

Processed 200 / 5828 artists...
Processed 400 / 5828 artists...
Processed 600 / 5828 artists...
Processed 800 / 5828 artists...
Processed 1000 / 5828 artists...
Processed 1200 / 5828 artists...
Processed 1400 / 5828 artists...
Processed 1600 / 5828 artists...
Processed 1800 / 5828 artists...
Processed 2000 / 5828 artists...
Processed 2200 / 5828 artists...
Processed 2400 / 5828 artists...
Processed 2600 / 5828 artists...
Processed 2800 / 5828 artists...
Processed 3000 / 5828 artists...
Processed 3200 / 5828 artists...
Processed 3400 / 5828 artists...
Processed 3600 / 5828 artists...
Processed 3800 / 5828 artists...
Processed 4000 / 5828 artists...
Processed 4200 / 5828 artists...
Processed 4400 / 5828 artists...
Processed 4600 / 5828 artists...
Processed 4800 / 5828 artists...
Processed 5000 / 5828 artists...
Processed 5200 / 5828 artists...
Processed 5400 / 5828 artists...
Processed 5600 / 5828 artists...
Processed 5800 / 5828 artists...
Processed 5828 / 5828 artists...
Graph built wi

In [36]:
# Basic stats
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

# Count mentions: who is mentioned most (incoming edges)
mention_count = Counter(v for _, v in G.edges())
top10 = mention_count.most_common(10)

print("Top-10 most-mentioned artists:")
for name, cnt in top10:
    print(f"{name}: mentioned by {cnt} pages")

Nodes: 5828
Edges: 10084
Top-10 most-mentioned artists:
30607: mentioned by 389 pages
8358: mentioned by 373 pages
297: mentioned by 358 pages
224: mentioned by 316 pages
84: mentioned by 312 pages
211: mentioned by 255 pages
223: mentioned by 242 pages
10590: mentioned by 242 pages
19179: mentioned by 240 pages
1060: mentioned by 225 pages


# Me thinking - the genre inclusion
A practice example is included below for future use

In [7]:
import pandas as pd
import networkx as nx
from itertools import combinations

# Sample data
df = pd.DataFrame({
    'Musician': ['Artist A', 'Artist B', 'Artist C', 'Artist D'],
    'Genres': [['rock', 'pop'], ['jazz/blues', 'rock'], ['classical'], ['rock', 'classical']]
})

df.set_index('Musician', inplace=True)

P = nx.Graph()
P.add_nodes_from(df.index)

# Add edges with shared genres attribute
for artist1, artist2 in combinations(df.index, 2):
    genres1 = set(df.loc[artist1, 'Genres'])
    genres2 = set(df.loc[artist2, 'Genres'])
    shared = genres1.intersection(genres2)
    if shared:
        P.add_edge(artist1, artist2, shared_genres=list(shared), weight=len(shared))

# Check edges and attributes
for u, v, attrs in P.edges(data=True):
    print(f"{u} <-> {v} shares {attrs['shared_genres']} (weight={attrs['weight']})")


Artist A <-> Artist B shares ['rock'] (weight=1)
Artist A <-> Artist D shares ['rock'] (weight=1)
Artist B <-> Artist D shares ['rock'] (weight=1)
Artist C <-> Artist D shares ['classical'] (weight=1)


In [8]:
# Find edges with 'rock' in their shared genres
rock_edges = [(u, v) for u, v, attrs in P.edges(data=True) if 'rock' in attrs['shared_genres']]

# Collect unique nodes connected by those edges
rock_nodes = set()
for u, v in rock_edges:
    rock_nodes.add(u)
    rock_nodes.add(v)

print(f"Number of nodes connected by 'rock' edges: {len(rock_nodes)}")
print("Nodes:", rock_nodes)


Number of nodes connected by 'rock' edges: 3
Nodes: {'Artist A', 'Artist B', 'Artist D'}


In [9]:
# or doing as an attribute of the node
for node in P.nodes():
    genres = df.loc[node, 'Genres']
    P.nodes[node]['influential'] = len(genres) > 1


# The Covered Relation Addition

In [37]:
# Merge on org_perf_id column in df_cov and perf_id in df_orig, keep only cov_art_id and org_art_id 
# Not merging on song title because songs can have the same names
df_merged = pd.merge(
    df_cov[['org_perf_id', 'cov_art_id']],
    df_orig[['perf_id', 'org_art_id']],
    left_on='org_perf_id',
    right_on='perf_id',
    how='inner'
)[['cov_art_id', 'org_art_id']]

# org_perf_id and perf_id
print(df_merged.head())
print(df_merged.shape)

# # Merge on org_perf_id column in df_cov and perf_id in df_orig, keep only cov_art_id and org_art_id 
# # Not merging on song title because songs can have the same names
# df_merged_2 = pd.merge(df_cov[['song_title', 'cov_art_id']], 
#                      df_orig[['song_title', 'org_art_id']], 
#                      on='song_title', 
#                      how='inner')

# # org_perf_id and perf_id
# print(df_merged.head())
# print(df_merged.shape)
# print(df_merged_2.shape)
# print(df_orig['song_title'].duplicated().sum())
# print(df_orig['perf_id'].duplicated().sum())
# print(df_cov['perf_id'].duplicated().sum())

  cov_art_id org_art_id
0      [879]        [1]
1   [5, 237]     [5483]
2        [7]        [6]
3       [10]        [8]
4        [9]        [8]
(367622, 2)


In [38]:
# Step 1: Parse stringified lists if needed
def parse_list(val):
    if isinstance(val, str):
        try:
            return ast.literal_eval(val)
        except:
            return [val]
    return val

df_merged['cov_art_id'] = df_merged['cov_art_id'].apply(parse_list)
df_merged['org_art_id'] = df_merged['org_art_id'].apply(parse_list)

# Step 2: Explode both columns
df_exploded = df_merged.explode('cov_art_id').explode('org_art_id').reset_index(drop=True)
print(df_exploded)


       cov_art_id org_art_id
0             879          1
1               5       5483
2             237       5483
3               7          6
4              10          8
...           ...        ...
482896      64925      11276
482897      35611      17062
482898      14339       2437
482899      14339      69125
482900        894       1389

[482901 rows x 2 columns]


In [39]:
edge_count = 0
for _, row in df_merged.iterrows():
    covering_artists = row['cov_art_id']
    original_artists = row['org_art_id']

    # Create edges for every pair (covering -> original)
    for cov_id in covering_artists:
        for org_id in original_artists:
            if cov_id != org_id: # avoid self-loop #and cov_id in G and org_id in G:
                if G.has_edge(cov_id, org_id):
                    edge_data = G[cov_id][org_id]
                    edge_data.setdefault('relation', set()).add('COVERED')
                    edge_data['weight'] = edge_data.get('weight', 0) + 1
                else:
                    G.add_edge(cov_id, org_id, relation={'COVERED'}, weight=1)
                edge_count += 1

In [40]:
import collections

# Step 2: Count how many times each artist is covered
cover_count = collections.Counter()
print(cover_count)
for u, v, data in G.edges(data=True):
    if data.get('relation') == 'COVERED':
        cover_count[v] += 1  # v = original artist who was covered

# Step 3: Get top 10 most covered artists
top_10_covered = cover_count.most_common(10)
print(top_10_covered)

# Step 4: Print them
for artist, count in top_10_covered:
    print(f"{artist}: covered {count} times")

Counter()
[]


In [41]:
print(f"Total nodes: {len(G.nodes)}")

# Find artist IDs/names that shouldn't coexist
mixed_nodes = [n for n in G.nodes if isinstance(n, int)]  # IDs
named_nodes = [n for n in G.nodes if isinstance(n, str)]  # Names

print(f"Nodes with IDs: {len(mixed_nodes)}")
print(f"Nodes with names: {len(named_nodes)}")

Total nodes: 65833
Nodes with IDs: 65833
Nodes with names: 0


In [15]:
# no real reason for this other than to convert everything to a string
# Merge on perf_id, keep only cov_art_id and org_art_id
top_10_artist_ids = [artist_id for artist_id, count in top_10_covered]


# Create the DataFrame
covered_top_10 = pd.DataFrame({
    'top_10_artist_ids': top_10_artist_ids,
    'artist_name': None
})


# Populate the artist_name column
for i, (artist_id, count) in enumerate(top_10_covered):
    try:
        artist_name = df_neo.loc[df_neo['artist_id'] == artist_id, 'common_name'].iloc[0]
    except IndexError:
        artist_name = "Unknown"
    covered_top_10.at[i, 'artist_name'] = artist_name

print(covered_top_10)

   top_10_artist_ids     artist_name
0                 41     The Beatles
1               4305  Duke Ellington
2                243     Bing Crosby
3                158       Bob Dylan
4               2232    Fred Astaire
5                319   Frank Sinatra
6               2223  Abbie Mitchell
7               1424    Judy Garland
8               5773     Joseph Mohr
9               5774    Franz Gruber


# Metrics

pagerank centrality metric
importance based on network structure

In [16]:
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 70189
Edges: 414401


In [18]:
# Compute PageRank
pagerank_scores = nx.pagerank(G, alpha=0.85) # is a default
top_10_pagerank = sorted(pagerank_scores.items(), key=lambda x: x[1], reverse=True)[:10]

# Extract artist IDs only (first value in each tuple)
top_10_artist_ids = [tup[0] for tup in top_10_pagerank]

# Create DataFrame
pagerank_top_10 = pd.DataFrame({
    'top_10_artist_ids': top_10_artist_ids,
    'artist_name': None
})

# Fill artist names
for i, artist_id in enumerate(covered_top_10['top_10_artist_ids']):
    try:
        artist_name = df_neo.loc[df_neo['artist_id'] == artist_id, 'common_name'].iloc[0]
    except IndexError:
        artist_name = "Unknown"
    pagerank_top_10.at[i, 'artist_name'] = artist_name

print(pagerank_top_10)


   top_10_artist_ids     artist_name
0                 41     The Beatles
1                158  Duke Ellington
2                616     Bing Crosby
3                206       Bob Dylan
4                243    Fred Astaire
5                 90   Frank Sinatra
6               4305  Abbie Mitchell
7              26565    Judy Garland
8               2223     Joseph Mohr
9                194    Franz Gruber


In-degree Centrality

In [19]:
# Compute in-degree centrality (for directed graphs)
in_degree_centrality = nx.in_degree_centrality(G)

# Sort and get top 10 nodes by in-degree centrality
in_degree_centrality_results= sorted(in_degree_centrality.items(), key=lambda x: x[1], reverse=True)[:10]

# Extract just the artist IDs into a list
in_degree_centrality_results= [artist_id for artist_id, score in in_degree_centrality_results]
print(in_degree_centrality_results)

# Create DataFrame
in_degree_top_10 = pd.DataFrame({
    'top_10_artist_ids': top_10_artist_ids,
    'artist_name': None
})

# Fill artist names
for i, artist_id in enumerate(covered_top_10['top_10_artist_ids']):
    try:
        artist_name = df_neo.loc[df_neo['artist_id'] == artist_id, 'common_name'].iloc[0]
    except IndexError:
        artist_name = "Unknown"
    in_degree_top_10.at[i, 'artist_name'] = artist_name

print(in_degree_top_10)

[41, 4305, 243, 158, 2232, 319, 2223, 1424, 5773, 5774]
   top_10_artist_ids     artist_name
0                 41     The Beatles
1                158  Duke Ellington
2                616     Bing Crosby
3                206       Bob Dylan
4                243    Fred Astaire
5                 90   Frank Sinatra
6               4305  Abbie Mitchell
7              26565    Judy Garland
8               2223     Joseph Mohr
9                194    Franz Gruber


Betweenness Centrality

Nodes that control flow / connectors

Example in Your Music Network Context
Betweenness could highlight artists who link different genres or scenes by covering songs from multiple communities.

Artists who bridge otherwise disconnected groups or influence multiple clusters.

They might not be the most covered (high in-degree) but have strategic importance connecting parts of the network.



In [20]:
# Approximate betweenness using a sample of nodes
# note: run time is insane otherwise (could take days), thus k set low
betweenness = nx.betweenness_centrality(G, k=100)  # try k=500 or lower

# Sort nodes by betweenness centrality descending and take top 10
top_10_betweenness = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:10]

# Extract just the artist IDs into a list
betweenness_centrality_results= [artist_id for artist_id, score in top_10_betweenness]
print(betweenness_centrality_results)

# Create DataFrame
btw_degree_top_10 = pd.DataFrame({
    'top_10_artist_ids': top_10_artist_ids,
    'artist_name': None
})

# Fill artist names
for i, artist_id in enumerate(covered_top_10['top_10_artist_ids']):
    try:
        artist_name = df_neo.loc[df_neo['artist_id'] == artist_id, 'common_name'].iloc[0]
    except IndexError:
        artist_name = "Unknown"
    btw_degree_top_10.at[i, 'artist_name'] = artist_name

print(btw_degree_top_10)

[41, 158, 243, 4305, 103, 1085, 319, 206, 2307, 1533]
   top_10_artist_ids     artist_name
0                 41     The Beatles
1                158  Duke Ellington
2                616     Bing Crosby
3                206       Bob Dylan
4                243    Fred Astaire
5                 90   Frank Sinatra
6               4305  Abbie Mitchell
7              26565    Judy Garland
8               2223     Joseph Mohr
9                194    Franz Gruber


# Network Description
The total network

In [21]:
# Total Network
# Basic stats
num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()
density = nx.density(G)

# Components (weakly connected components for directed graph)
num_components = nx.number_weakly_connected_components(G)

# Average shortest path length (only valid on strongly connected graphs or components)
# We'll use the largest weakly connected component
largest_cc = max(nx.weakly_connected_components(G), key=len)
G_sub = G.subgraph(largest_cc)
try:
    avg_shortest_path = nx.average_shortest_path_length(G_sub)
except:
    avg_shortest_path = "N/A (graph not connected)"

# Create a table
metrics = {
    "Metric": [
        "Number of vertices (artists)",
        "Number of edges (song covers)",
        "Number of components",
        "Average shortest path length",
        "Density"
    ],
    "Value": [
        num_nodes,
        num_edges,
        num_components,
        avg_shortest_path,
        density
    ]
}

metrics_df = pd.DataFrame(metrics)
print(metrics_df.to_string(index=False))

                       Metric                     Value
 Number of vertices (artists)                     70189
Number of edges (song covers)                    414401
         Number of components                      3220
 Average shortest path length N/A (graph not connected)
                      Density                  0.000084


The covered relation

In [22]:
# The Network: covered relation
# Basic stats
covered_edges = [(u, v) for u, v, d in G.edges(data=True) if d.get('relation') == 'COVERED']
# Extract artist IDs only (first value in each tuple)
num_covered_edges = len([tup[0] for tup in covered_edges])

# Get unique nodes involved in these edges
covered_nodes = len(set([u for u, v in covered_edges] + [v for u, v in covered_edges]))


# Compute density
G_covered = G.edge_subgraph(covered_edges).copy()
density_covered = nx.density(G_covered)

# Components (weakly connected components for directed graph)
num_components_covered = nx.number_weakly_connected_components(G_covered)

# Average shortest path length (only valid on strongly connected graphs or components)
# We'll use the largest weakly connected component
largest_cc_covered = max(nx.weakly_connected_components(G_covered), key=len)
G_sub_covered = G.subgraph(largest_cc_covered)
try:
    avg_shortest_path_covered = nx.average_shortest_path_length(G_sub_covered)
except:
    avg_shortest_path_covered = "N/A (graph not connected)"

# Create a table
metrics_covered = {
    "Metric": [
        "Number of vertices (artists)",
        "Number of edges (song covers)",
        "Number of components",
        "Average shortest path length",
        "Density"
    ],
    "Value": [
        covered_nodes,
        num_covered_edges,
        num_components_covered,
        avg_shortest_path_covered,
        density_covered
    ]
}

metrics_df_covered = pd.DataFrame(metrics_covered)
print(metrics_df_covered.to_string(index=False))

                       Metric                     Value
 Number of vertices (artists)                     64361
Number of edges (song covers)                    404317
         Number of components                       189
 Average shortest path length N/A (graph not connected)
                      Density                  0.000098


The Network: wiki relation

In [23]:
# The Network: wiki relation
# Basic stats
mentions_edges = [(u, v) for u, v, d in G.edges(data=True) if d.get('relation') == 'MENTIONS']
# Extract artist IDs only (first value in each tuple)
num_mentions_edges = len([tup[0] for tup in mentions_edges])

# Get unique nodes involved in these edges
mentions_nodes = len(set([u for u, v in mentions_edges] + [v for u, v in mentions_edges]))


# Compute density
G_mentions = G.edge_subgraph(mentions_edges).copy()
density_mentions = nx.density(G_mentions)

# Components (weakly connected components for directed graph)
num_components_mentions = nx.number_weakly_connected_components(G_mentions)

# Average shortest path length (only valid on strongly connected graphs or components)
# We'll use the largest weakly connected component
largest_cc_mentions = max(nx.weakly_connected_components(G_mentions), key=len)
G_sub_mentions = G.subgraph(largest_cc_mentions)
try:
    avg_shortest_path_mentions = nx.average_shortest_path_length(G_sub_mentions)
except:
    avg_shortest_path_mentions = "N/A (graph not connected)"

# Create a table
metrics_mentions = {
    "Metric": [
        "Number of vertices (artists)",
        "Number of edges (song covers)",
        "Number of components",
        "Average shortest path length",
        "Density"
    ],
    "Value": [
        mentions_nodes,
        num_mentions_edges,
        num_components_mentions,
        avg_shortest_path_mentions,
        density_covered
    ]
}

metrics_df_mentions = pd.DataFrame(metrics_mentions)
print(metrics_df_mentions.to_string(index=False))

                       Metric                     Value
 Number of vertices (artists)                      2809
Number of edges (song covers)                     10084
         Number of components                        12
 Average shortest path length N/A (graph not connected)
                      Density                  0.000098
